# 06 — Model: Random Forest

Requires `train_features.csv` / `test_features.csv` from **01_feature_engineering.ipynb**.
No extra installs needed.

A bagging-based ensemble rather than boosting — trained on independently bootstrapped samples with random
feature subsets at each split. It won't beat the boosted models individually, but bagging errors are
decorrelated from boosting errors, so it tends to add real value to the blend even at a lower solo AUC.

Categoricals are label-encoded here (sklearn's forests don't take raw categories the way LightGBM/CatBoost
do). Saves `oof_rf.csv` and `test_pred_rf.csv` for the ensembling notebook.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

DATA_DIR = "."   # <-- folder with train_features.csv / test_features.csv from notebook 01
N_FOLDS = 5
SEED = 42

train_fe = pd.read_csv(f"{DATA_DIR}/train_features.csv")
test_fe = pd.read_csv(f"{DATA_DIR}/test_features.csv")

num_cols = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours',
            'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time',
            'notif_per_hour', 'app_per_hour', 'mins_per_appopen', 'mins_per_notif', 'productivity',
            'sleep_screen_sum', 'nonscreen_hours', 'screen_plus_weekend', 'screen_sleep_ratio',
            'sm_ratio', 'game_ratio', 'work_ratio', 'screen_minus_work', 'weekday_weekend_ratio',
            'screen_x_sm', 'screen_x_weekend', 'sm_x_weekend', 'screen_x_sleep']
cat_cols = ['gender', 'stress_level', 'academic_work_impact']
feat_cols = num_cols + cat_cols

X = train_fe[feat_cols].copy()
Xtest = test_fe[feat_cols].copy()
y = train_fe['addicted_label'].values

for c in cat_cols:
    X[c] = X[c].astype('category')
    Xtest[c] = Xtest[c].astype('category')

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))
print(X.shape, Xtest.shape)

from sklearn.ensemble import RandomForestClassifier
import time

X_rf = X.copy()
Xtest_rf = Xtest.copy()
for c in cat_cols:
    codes, uniques = pd.factorize(pd.concat([X_rf[c], Xtest_rf[c]]), use_na_sentinel=True)
    X_rf[c] = codes[:len(X_rf)]
    Xtest_rf[c] = codes[len(X_rf):]
for c in num_cols:
    med = X_rf[c].median()
    X_rf[c] = X_rf[c].fillna(med)
    Xtest_rf[c] = Xtest_rf[c].fillna(med)


(691369, 30) (296302, 30)


In [ ]:
oof_rf = np.zeros(len(X))
test_rf = np.zeros(len(Xtest))

t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(folds):
    model = RandomForestClassifier(
        n_estimators=400, max_depth=18, min_samples_leaf=20, max_features='sqrt',
        n_jobs=-1, random_state=SEED
    )
    model.fit(X_rf.iloc[tr_idx], y[tr_idx])
    p_va = model.predict_proba(X_rf.iloc[va_idx])[:, 1]
    oof_rf[va_idx] = p_va
    test_rf += model.predict_proba(Xtest_rf)[:, 1] / N_FOLDS
    print(f"fold {fold} auc={roc_auc_score(y[va_idx], p_va):.5f}  ({time.time()-t0:.0f}s elapsed)")

print("Random Forest OOF AUC:", roc_auc_score(y, oof_rf))


In [ ]:
pd.DataFrame({'id': train_fe['id'], 'oof_pred': oof_rf}).to_csv(f"{DATA_DIR}/oof_rf.csv", index=False)
pd.DataFrame({'id': test_fe['id'], 'test_pred': test_rf}).to_csv(f"{DATA_DIR}/test_pred_rf.csv", index=False)
print("saved oof_rf.csv and test_pred_rf.csv")
